In [2]:
# parse_orders.py (paste this into ONE Jupyter notebook cell in parse.ipynb)
# ------------------------------------------------------------
# This solution follows the assignment rule: read file as TEXT and extract with REGEX (no csv module).
# It:
# 1) extracts order numbers, product codes, prices, dates (as raw lists)
# 2) builds per-order records (so you can filter/find max/min)
# 3) filters orders priced > $500
# 4) converts dates to DD/MM/YYYY using re.sub()
# 5) finds orders with the highest item count
# 6) finds the cheapest order(s)
#
# IMPORTANT:
# - Regex patterns depend on how orders.csv is formatted.
# - If your file’s date format or column order differs, adjust the ORDER_LINE_REGEX below.

import re
from dataclasses import dataclass
from typing import List, Optional, Tuple


# ---------- 1) Read the CSV file as raw text ----------
with open("./csv/orders.csv", "r", encoding="utf-8") as f_in:
    text = f_in.read()

lines = [ln for ln in text.splitlines() if ln.strip()]


# ---------- 2) Raw extraction tasks (regex on full text) ----------
# A) "Order numbers": we try to avoid grabbing date parts/prices by looking for an "order" label first.
# If your file doesn't label the order number, see fallback regex below.
ORDER_NO_REGEX = r"(?i)\border(?:_?\s*id|_?\s*no|_?\s*number)?\b[^0-9]*([0-9]+)"

# Fallback (use ONLY if your file has no "order" label and order numbers are a clear field):
# ORDER_NO_REGEX = r"(?m)^\s*([0-9]+)\s*,"

# B) "Product codes": common pattern like ABC123 or PRD-1001, SKU-999, etc.
PRODUCT_CODE_REGEX = r"\b[A-Z]{2,}[A-Z0-9-]*\b"

# C) "Prices": typical $123.45 (captures number part, we convert to float)
PRICE_REGEX = r"\$(\d+(?:\.\d{2})?)"

# D) "Order dates": assume ISO format YYYY-MM-DD (very common in assignments)
DATE_REGEX = r"\b\d{4}-\d{2}-\d{2}\b"

order_numbers_raw = re.findall(ORDER_NO_REGEX, text)
product_codes_raw = re.findall(PRODUCT_CODE_REGEX, text)
prices_raw = re.findall(PRICE_REGEX, text)
dates_raw = re.findall(DATE_REGEX, text)

# Convert prices to floats
prices_float = [float(p) for p in prices_raw]


print("RAW EXTRACTIONS")
print("---------------")
print(f"Order numbers (raw): {order_numbers_raw[:10]}{' ...' if len(order_numbers_raw) > 10 else ''}")
print(f"Product codes (raw): {product_codes_raw[:10]}{' ...' if len(product_codes_raw) > 10 else ''}")
print(f"Prices (raw): {prices_raw[:10]}{' ...' if len(prices_raw) > 10 else ''}")
print(f"Dates (raw): {dates_raw[:10]}{' ...' if len(dates_raw) > 10 else ''}")
print()


# ---------- 3) Build structured orders per line (recommended for filtering/min/max) ----------
# This regex assumes each order line looks roughly like:
#   <order_no>,<product_code>,$<price>,<YYYY-MM-DD>,<items>
#
# Example:
#   10023,PRD-9001,$799.99,2025-01-05,3
#
# If your column order differs, adjust this ONE regex.
ORDER_LINE_REGEX = re.compile(
    r"""^\s*
        (?P<order_no>\d+)\s*,\s*
        (?P<product_code>[A-Z0-9-]+)\s*,\s*
        \$\s*(?P<price>\d+(?:\.\d{2})?)\s*,\s*
        (?P<date>\d{4}-\d{2}-\d{2})\s*,\s*
        (?P<items>\d+)
        \s*$""",
    re.VERBOSE,
)

@dataclass(frozen=True)
class Order:
    order_no: str
    product_code: str
    price: float
    date: str          # stored as YYYY-MM-DD initially
    items: int

def parse_orders_from_lines(lines: List[str]) -> List[Order]:
    orders: List[Order] = []
    for ln in lines:
        m = ORDER_LINE_REGEX.match(ln)
        if not m:
            continue
        orders.append(
            Order(
                order_no=m.group("order_no"),
                product_code=m.group("product_code"),
                price=float(m.group("price")),
                date=m.group("date"),
                items=int(m.group("items")),
            )
        )
    return orders

orders = parse_orders_from_lines(lines)

print("STRUCTURED PARSE")
print("---------------")
print(f"Parsed {len(orders)} orders (lines matched ORDER_LINE_REGEX).")
if len(orders) == 0:
    print("No lines matched. Your orders.csv likely has a different column order/format.")
    print("Fix: paste one example line from csv/orders.csv and adjust ORDER_LINE_REGEX.")
print()


# ---------- 4) Required answers using the structured orders ----------
# 4.1 Extract all order numbers
order_numbers = [o.order_no for o in orders]

# 4.2 Extract all product codes
product_codes = [o.product_code for o in orders]

# 4.3 Extract all prices
prices = [o.price for o in orders]

# 4.4 Extract all order dates
order_dates = [o.date for o in orders]


print("ANSWERS")
print("-------")
print("All order numbers:", order_numbers)
print("All product codes:", product_codes)
print("All prices:", prices)
print("All order dates:", order_dates)
print()


# ---------- 5) Find all orders for products priced over $500 ----------
orders_over_500 = [o for o in orders if o.price > 500]
print("Orders priced over $500:")
for o in orders_over_500:
    print(o)
print()


# ---------- 6) Change date format to DD/MM/YYYY using re.sub() ----------
# Convert each date string using capture groups: YYYY-MM-DD -> DD/MM/YYYY
def to_dd_mm_yyyy(date_yyyy_mm_dd: str) -> str:
    return re.sub(r"^(\d{4})-(\d{2})-(\d{2})$", r"\3/\2/\1", date_yyyy_mm_dd)

converted_dates = [to_dd_mm_yyyy(o.date) for o in orders]
print("Dates converted to DD/MM/YYYY:", converted_dates)
print()


# ---------- 7) Find all orders with the highest number of ordered items ----------
if orders:
    max_items = max(o.items for o in orders)
    highest_item_orders = [o for o in orders if o.items == max_items]
    print(f"Highest item count = {max_items}")
    print("Orders with the highest number of ordered items:")
    for o in highest_item_orders:
        print(o)
    print()
else:
    print("No structured orders parsed; cannot compute highest item count.\n")


# ---------- 8) Find the cheapest order(s) ----------
if orders:
    min_price = min(o.price for o in orders)
    cheapest_orders = [o for o in orders if o.price == min_price]
    print(f"Cheapest price = {min_price}")
    print("Cheapest order(s):")
    for o in cheapest_orders:
        print(o)
    print()
else:
    print("No structured orders parsed; cannot compute cheapest order(s).\n")


FileNotFoundError: [Errno 2] No such file or directory: './csv/orders.csv'